# CI/CD as a quality gate — including the eval gate

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 36 §36.3, §36.5 · type: concept-lab*

**One-line promise:** model a CI/CD pipeline as a *quality gate* — lint, types, tests, **and evals** — that blocks a merge on regression, then simulate a canary that auto-rolls-back on a metric dip. Everything runs **offline, free, with no CI runner, no cloud, and no spend**.

This is a concept lab: many small cells, predict-then-run beats, pure-Python simulation. Picking up from [`36-01-terraform-plan-locally.ipynb`](./36-01-terraform-plan-locally.ipynb) — where you produced a reviewable `plan` — this notebook is the *pipeline that runs that plan for you*, only after the change has earned its way through the gate.

## 🧠 Why this matters

CI/CD automates the path from a git push to running software — but automation alone is not the point. A pipeline that *automates* a deploy without *gating* on quality just ships your bugs faster. The senior framing is: **CI/CD is a quality gate.** On every change, CI runs lint, type-checks, and tests — and for agentic systems, **evals** (Part VI) — so a quality regression blocks the merge exactly like a failing unit test. Only what passes the gate flows to **CD**: build the image, apply the IaC (36-01), roll out to staging, then a canary to prod.

The AI-specific addition is the **eval gate**. A prompt or model change can pass every unit test and still quietly degrade answer quality — unit tests check code paths, not whether the agent is still *good*. An eval suite scores the agent on a fixed set of cases; if the score drops below threshold, the merge is blocked. Pair that with **progressive delivery** (Ch 28) — a canary watching live metrics, auto-rolling-back on a dip — and shipping changes to an agentic system becomes *boring*, which for production software is exactly the goal. See §36.3.

## Objectives & prereqs

**By the end you can:**
- Model a CI/CD pipeline as an ordered list of **stages**, each pass/fail, that short-circuits on the first failure — the §36.3 shape.
- Implement the **eval gate**: load eval scores, compare a candidate against a threshold, and block the merge on a regression.
- Predict whether a given change merges or is blocked — *before* running the gate.
- Simulate **progressive delivery**: a canary watching a metric and auto-rolling-back on a dip (a tiny state machine).
- Articulate the §36.5 senior lens: what **platform engineering** buys an organization.

**Prereqs:** 36-01 (you produced the `plan` this pipeline applies) · Ch 7 (testing & CI) · Part VI (evals — the eval-gate concept). No tools beyond Python are needed.

**Packages:** standard library only (plus `python-dotenv` from `requirements.txt`). There is **no model call and no network** — the pipeline, the eval scores, and the canary are all simulated locally.

In [ ]:
# --- Setup: imports, env, MOCK switch, determinism -------------------------
# Stdlib only (+ python-dotenv). NO API calls, NO network, NO cloud. The
# pipeline / eval gate / canary are simulated in pure Python so this runs
# identically and free for every reader.
import os
import json
import random
from pathlib import Path
from dataclasses import dataclass

try:
    from dotenv import load_dotenv
    load_dotenv()  # reads a local .env if present; never hardcode keys
except ImportError:
    pass

# MOCK=1 (default) = fully offline, deterministic, $0. There is no live path to
# enable here: a real CI runner is a heavy, stateful, cloud-bound thing, so this
# lab ALWAYS simulates. We keep the switch for consistency with the other
# notebooks and to make the "no live spend" contract explicit.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Seed every stochastic beat (the canary metric stream) so the lesson is
# reproducible. Real evals/canaries are noisy; we fix the noise for teaching.
SEED = 36
random.seed(SEED)

# Tiny committed fixture: baseline vs candidate eval scores for the gate demo.
DATA = Path("data") / "eval-scores.json"
print(f"MOCK mode      : {MOCK}  (offline, free, no CI runner, no cloud)")
print(f"random seed    : {SEED}  (canary metric stream is reproducible)")
print(f"eval fixture   : {DATA}  exists={DATA.exists()}")
if not DATA.exists():
    print("  NOTE: run this notebook from its own folder so data/ resolves.")

## 1 · 🧠 Mental model: the pipeline is a gate, not a conveyor belt

Picture the pipeline as a sequence of **stages**, each of which can pass or fail:

```
push → [ lint ] → [ type-check ] → [ unit tests ] → [ EVAL GATE ] → [ build image ]
     → [ apply IaC ] → [ deploy staging ] → [ canary → prod ]
```

**CI** is everything up to and including the eval gate — cheap checks that run on *every* change. **CD** is everything after — it only runs for a change that already passed CI. The critical property: the pipeline **short-circuits on the first failure**. A failing lint stage means type-check, tests, and evals never run, and nothing is built or shipped. That short-circuit is what makes the pipeline a *gate* — a bad change cannot reach production by sailing past a check it should have failed.

In [ ]:
# A tiny offline pipeline simulator. A Stage is a name + a function that returns
# (passed, detail). The runner executes stages in order and STOPS at the first
# failure -- exactly how a real CI/CD pipeline gates (§36.3).
@dataclass
class StageResult:
    name: str
    passed: bool
    detail: str


@dataclass
class Stage:
    name: str
    check: object  # callable(change: dict) -> (bool, str)
    kind: str = "ci"  # "ci" runs on every change; "cd" only after CI passes


def run_pipeline(stages, change):
    """Run stages in order; short-circuit on first failure. Returns results."""
    results = []
    for st in stages:
        passed, detail = st.check(change)
        results.append(StageResult(st.name, passed, detail))
        if not passed:
            # Gate slammed shut: nothing after this runs, nothing ships.
            break
    return results


def render(results, stages):
    shipped = all(r.passed for r in results) and len(results) == len(stages)
    for r in results:
        mark = "PASS" if r.passed else "FAIL"
        print(f"  [{mark}] {r.name:<16} {r.detail}")
    skipped = len(stages) - len(results)
    if skipped:
        print(f"  [skip] {skipped} later stage(s) skipped (gate blocked the change)")
    print(f"\n  RESULT: {'SHIPPED' if shipped else 'BLOCKED -- nothing deployed'}")
    return shipped


print("simulator ready: Stage, run_pipeline(), render()")

## 2 · The CI stages, and a "good" change

Below, each CI stage is a small check over a `change` dict (think: the metadata CI would compute for a pull request). The cheap, deterministic checks — lint, type-check, unit tests — come first; the eval gate (section 3) comes after them, because evals are the most expensive check and there's no point evaluating code that doesn't even compile.

We run the pipeline first on a **clean change** that passes everything.

In [ ]:
# Cheap CI checks first. Each reads precomputed signals off the `change` dict
# the way real CI reads tool exit codes (ruff/black, mypy, pytest).
def lint_check(change):
    n = change.get("lint_errors", 0)
    return n == 0, f"ruff/black: {n} issue(s)"


def type_check(change):
    n = change.get("type_errors", 0)
    return n == 0, f"mypy: {n} error(s)"


def unit_tests(change):
    failed = change.get("tests_failed", 0)
    total = change.get("tests_total", 0)
    return failed == 0, f"pytest: {total - failed}/{total} passed"


CI_STAGES = [
    Stage("lint", lint_check, "ci"),
    Stage("type-check", type_check, "ci"),
    Stage("unit-tests", unit_tests, "ci"),
]

# A clean change: no lint/type errors, all tests green.
good_change = {
    "label": "pr-241: tighten retrieval top_k",
    "lint_errors": 0, "type_errors": 0,
    "tests_total": 128, "tests_failed": 0,
}

print(f"Change under test: {good_change['label']}\n")
render(run_pipeline(CI_STAGES, good_change), CI_STAGES)

## 3 · 🔧 The eval gate (§36.3 tip · Part VI)

Here is the AI-specific stage. Unit tests confirm the code runs; **evals confirm the agent is still good.** The gate loads an eval score for the candidate and blocks the merge if it falls below a threshold — just like a failing test, but measuring *quality* instead of *correctness*.

Our fixture `data/eval-scores.json` holds a baseline and two candidates:

- `good-change` — a retrieval tweak that *raises* accuracy to **0.88**.
- `regression` — a "make it concise" system-prompt rewrite that *drops* accuracy to **0.74**.

The threshold is **0.80**.

> 🔮 **Predict.** The `regression` candidate scores **0.74** against a **0.80** threshold. When we add the eval gate to the pipeline and run the `regression` change through it, does the change **merge** or get **blocked** — and which stages run before the gate decides? Write your guess, then run the next two cells.

In [ ]:
# Load the committed fixture. baseline + two candidates, one threshold.
scores = json.loads(DATA.read_text(encoding="utf-8"))

threshold = scores["threshold"]
baseline = scores["baseline"]["score"]
print(f"suite     : {scores['suite']}  (metric: {scores['metric']})")
print(f"threshold : {threshold}")
print(f"baseline  : {baseline}  ({scores['baseline']['label']})\n")
for key, cand in scores["candidates"].items():
    delta = cand["score"] - baseline
    print(f"  {key:<12} score={cand['score']:.2f}  d_baseline={delta:+.2f}  {cand['label']}")

In [ ]:
# The eval gate: a CI stage that blocks the merge when score < threshold.
# This is the one stage a "passes all tests" change can still fail (§36.3).
def make_eval_gate(all_scores):
    th = all_scores["threshold"]

    def eval_gate(change):
        # The change names which candidate's eval result it carries.
        cand = all_scores["candidates"][change["eval_candidate"]]
        score = cand["score"]
        ok = score >= th
        verdict = "above" if ok else "BELOW"
        return ok, f"eval: {score:.2f} {verdict} threshold {th:.2f}"

    return eval_gate


# Full CI pipeline now ends with the eval gate.
CI_WITH_EVAL = CI_STAGES + [Stage("eval-gate", make_eval_gate(scores), "ci")]

# The regression change: code is fine (lint/types/tests all pass) but the
# prompt rewrite tanks eval accuracy. The gate must catch what tests cannot.
regression_change = {
    "label": scores["candidates"]["regression"]["label"],
    "lint_errors": 0, "type_errors": 0,
    "tests_total": 128, "tests_failed": 0,
    "eval_candidate": "regression",
}

print(f"Change under test: {regression_change['label']}\n")
shipped = render(run_pipeline(CI_WITH_EVAL, regression_change), CI_WITH_EVAL)
print(f"\n=> merged? {shipped}   (predict matched? the eval gate is the blocker)")

**What you just saw.** Lint, type-check, and unit tests all passed — the *code* is fine. The change was blocked anyway, by the **eval gate**, because answer accuracy fell from 0.86 to 0.74, under the 0.80 bar. This is the failure mode the eval gate exists for: a prompt or model edit that no unit test would catch silently degrading the agent. Without this stage, the regression merges and your users feel it in production before you do.

Now contrast with the *good* change — same code-quality checks, but its eval score (0.88) clears the bar, so it sails through.

In [ ]:
# The good change carries the "good-change" eval result (0.88 >= 0.80).
good_change_full = dict(good_change, eval_candidate="good-change")

print(f"Change under test: {good_change_full['label']}\n")
shipped = render(run_pipeline(CI_WITH_EVAL, good_change_full), CI_WITH_EVAL)
print(f"\n=> merged? {shipped}   (improves accuracy 0.86 -> 0.88, clears the gate)")

## 4 · Progressive delivery: a canary that rolls back on a dip (Ch 28)

Passing CI gets a change *merged and built*; it does not mean it's safe in production. **Progressive delivery** ships it gradually: route a small slice of traffic to the new version (the **canary**), watch a live metric, and **roll back automatically** if the metric dips — long before a full rollout. This is a tiny state machine, not a real deploy.

Our canary watches a **success rate** and trips if it drops below a floor for two consecutive readings (one bad sample could be noise; a sustained dip is a real regression).

> 🔮 **Predict.** The metric stream below starts healthy (~0.99) and then a bad deploy drags it down. With a floor of **0.95** and a "two consecutive breaches" rule, will the canary **promote** the release to full traffic, or **roll it back**? Predict, then run.

In [ ]:
# A small canary state machine: WATCHING -> (PROMOTED | ROLLED_BACK).
# It reads a stream of success-rate samples and trips after `patience`
# consecutive readings below `floor`. Seeded above, so the stream is fixed.
@dataclass
class Canary:
    floor: float = 0.95     # minimum acceptable success rate
    patience: int = 2       # consecutive breaches before rollback
    breaches: int = 0
    state: str = "WATCHING"

    def observe(self, success_rate):
        if self.state != "WATCHING":
            return self.state
        if success_rate < self.floor:
            self.breaches += 1
            if self.breaches >= self.patience:
                self.state = "ROLLED_BACK"
        else:
            self.breaches = 0  # recovered; one dip was just noise
        return self.state


def gen_metric_stream(n_healthy=4, n_bad=4):
    """Healthy samples near 0.99, then a bad-deploy dip near 0.90 (+ noise)."""
    stream = []
    for _ in range(n_healthy):
        stream.append(round(min(1.0, random.gauss(0.992, 0.004)), 3))
    for _ in range(n_bad):
        stream.append(round(max(0.0, random.gauss(0.90, 0.01)), 3))
    return stream


canary = Canary(floor=0.95, patience=2)
stream = gen_metric_stream()

print(f"canary: floor={canary.floor}  rollback after {canary.patience} consecutive breaches\n")
for i, sr in enumerate(stream, 1):
    state = canary.observe(sr)
    flag = "" if sr >= canary.floor else f"  <- breach #{canary.breaches}"
    print(f"  t{i}: success_rate={sr:.3f}  state={state}{flag}")
    if state == "ROLLED_BACK":
        print("\n  ROLLED BACK: metric stayed below floor -- prod keeps the old version")
        break
else:
    print("\n  PROMOTED: metric healthy throughout -- canary goes to full traffic")

**What you just saw.** Four healthy readings, then the bad deploy's success rate fell below the 0.95 floor; after the *second* consecutive breach the canary tripped to `ROLLED_BACK` and traffic stayed on the old version — no human paged at 3 a.m. The `patience` of 2 is what keeps a single noisy sample from triggering a needless rollback while still catching a sustained regression fast.

## ⚠️ Pitfall: a pipeline that automates but doesn't *gate*

The seductive mistake is to build a pipeline that *runs* lint/tests/deploy automatically but treats the results as advisory — logging a red check and shipping anyway, or omitting the eval gate so quality regressions have no stage that can stop them. That pipeline makes you ship **faster**, including your regressions. The fix is to make passing the checks a **hard precondition** for the next stage: tests, types, *and* evals must be green together, the pipeline short-circuits on any failure, and the canary can veto a rollout the offline checks couldn't foresee. Automation without gating is just a faster way to break production.

In [ ]:
# Make the "automates but doesn't gate" bug concrete: a runner that records
# failures but ships regardless. Same regression change -- opposite outcome.
def run_pipeline_ungated(stages, change):
    """ANTI-PATTERN: runs every stage, ignores failures, ships anyway."""
    results = [StageResult(st.name, *st.check(change)) for st in stages]
    return results  # note: NO short-circuit, and the caller ignores .passed


results = run_pipeline_ungated(CI_WITH_EVAL, regression_change)
for r in results:
    mark = "PASS" if r.passed else "FAIL"
    print(f"  [{mark}] {r.name:<16} {r.detail}")
print("\n  UNGATED RESULT: shipped anyway -- the eval regression reached prod")
print("  (the gate FOUND the problem; the broken pipeline just didn't ACT on it)")
print("\nFix: block on the first failure AND require lint+types+tests+evals green together.")

## 🎯 Senior lens (§36.5)

Scale this discipline up and it becomes **platform engineering**: you stop hand-building one pipeline for one service and start building the IaC, pipelines, and paved-road tooling that let *every* team ship safely without reinventing it. The eval gate, the canary-with-rollback, the secrets-by-ARN rule from 36-01 — a platform team codifies these as the **default** so an app team gets them for free by adopting the template, not by remembering to wire them up.

The leverage shift is the whole point of the staff/architect arc: early on, your impact is deploying *your* service well; later, it's making it easy and safe for the *organization* to deploy well — baking in the right defaults for security, observability, cost, and rollouts. The highest form of infrastructure work isn't running systems; it's building the system that lets a whole organization run systems. A gated pipeline is where that philosophy first becomes executable.

## Recap

- **CI/CD is a quality gate, not a conveyor belt** — the pipeline **short-circuits on the first failure**, so a bad change can't reach prod past a check it should fail (§36.3).
- **CI** runs on every change (lint → types → tests → **eval gate**); **CD** ships only what passes (build → apply IaC → staging → canary → prod).
- **The eval gate** catches what unit tests can't: a prompt/model change that passes all tests yet drops answer quality below threshold (here 0.74 vs a 0.80 bar) is **blocked** (Part VI).
- **Progressive delivery** = a canary watching a live metric that **auto-rolls-back** on a sustained dip; `patience` guards against rolling back on noise (Ch 28).
- **The trap:** automating without gating just ships regressions faster — the fix is to require tests *and* types *and* evals green together.
- **§36.5 platform engineering:** codify these defaults as a paved road so the whole org ships safely — that's where your leverage goes as you grow.

## Exercises

Predict the result before running each.

1. **Reorder the gate.** Move `eval-gate` *before* `unit-tests` in `CI_WITH_EVAL` and re-run the `regression` change. Does the outcome change? Should the cheapest checks run first, or the most decisive? Argue the trade-off.
2. **Tune the threshold.** Lower the eval `threshold` to `0.70` (in memory — don't edit the fixture) and re-run the `regression` change. Does it merge now? What real-world risk does loosening the bar trade away?
3. **Add a stage.** Add a `security-scan` CI stage (e.g. fails if `change["high_cves"] > 0`) and a change that passes everything *except* the scan. Predict which stages run before it blocks.
4. **Tune the canary.** Change `patience` to `1` and re-run section 4, then to `3`. How does sensitivity to a single noisy sample change? When would each setting be the right call?
5. **Wire 36-01 in.** Sketch (in a markdown cell) where `terraform plan` (review) and `terraform apply` (from 36-01) slot into the CD half of this pipeline, and which human approval gates them.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


In [ ]:
# Exercise 5 -- your sketch here (or use a markdown cell)


## Next

- ⬅️ **Previous notebook:** [`36-01-terraform-plan-locally.ipynb`](./36-01-terraform-plan-locally.ipynb) — the reproducible `plan` this pipeline applies for you.
- 🧪 **Eval harness (the gate's real engine):** the offline scores here stand in for [`blueprints/eval-harness/`](../../../blueprints/eval-harness/) from Part VI — plug a real suite in to compute the score the gate reads.
- 📦 **Templates:** this pipeline applies the module from [`templates/terraform-module/`](../../../templates/terraform-module/) and reinforces the CI template from Ch 7.
- 🏗️ **Capstone:** the gated pipeline deploys [`capstone/infra/`](../../../capstone/infra/) — the full Chapter 33 architecture as Terraform — closing Part VIII. Advances checkpoint `checkpoints/ch36-infra-as-code`.
- 📘 See the book **§36.3** (CI/CD + the eval gate) and **§36.5** (platform engineering).